# 💊 Clinical-Grade Drug Interaction Reasoning — RAG Retrieval Pipeline  
<span style="color:red">by Ridwan Oladipo, MD | Medical AI Specialist</span>  

Production-grade **Retrieval-Augmented Generation (RAG)** system bridging curated DrugBank–RxNorm knowledge with intelligent clinical inference:  
- **Hybrid 3-tier architecture** (direct RxCUI lookup → FAISS semantic search → confidence-gated abstain)  
- **Cosine-normalized vector index** (170K interactions × 3072-dim embeddings via text-embedding-3-large)  
- **Dynamic lexical pre-filtering** for accelerated semantic recall  
- **Polypharmacy-aware reasoning** (N drugs → pairwise expansion + aggregated risk synthesis)  
- **Tiered confidence scoring** (Tier 1: 1.0 exact, Tier 2: 0.6-0.9 semantic, Tier 3: escalate)
- **Multi-cloud deployment ready** (AWS Bedrock Titan V2 tested for enterprise HIPAA compliance)  

🚀 Handles complex queries like *"Warfarin, Aspirin, Ibuprofen"* by retrieving all pairwise evidence and enabling GPT-5 synthesis for cumulative risk—solving polypharmacy challenges that deterministic systems cannot address.  

>⚕️ **Pharmacological precision meets AI reasoning** — delivering evidence-based decision support for safe medication management.

## 📦 Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
import time
import json
import boto3
from itertools import combinations
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import faiss

sys.path.append('../src')
from ingest import normalize_to_ingredient_rxcui

print("✅ Imports loaded")

✅ Imports loaded


## 📂 Load Knowledge Base

In [2]:
kb_df = pd.read_csv('../data/processed_interactions_kb.csv')
kb_df['pair_key'] = kb_df['pair_key'].apply(eval)

print(f"✅ Loaded knowledge base:")
print(f"   Total interactions: {len(kb_df):,}")
print(f"   Unique drug pairs: {kb_df['pair_key'].nunique():,}")
print(f"\nSample:")
print(kb_df[['Drug 1', 'Drug 2', 'Interaction Description', 'pair_key']].head())

✅ Loaded knowledge base:
   Total interactions: 170,782
   Unique drug pairs: 170,782

Sample:
                Drug 1       Drug 2  \
0           Trioxsalen  Verteporfin   
1  Aminolevulinic acid  Verteporfin   
2     Titanium dioxide  Verteporfin   
3     Tiaprofenic acid  Verteporfin   
4          Cyamemazine  Verteporfin   

                             Interaction Description          pair_key  
0  Trioxsalen may increase the photosensitizing a...   (10844, 118886)  
1  Aminolevulinic acid may increase the photosens...     (118886, 683)  
2  Titanium dioxide may increase the photosensiti...   (118886, 38323)  
3  Tiaprofenic acid may increase the photosensiti...  (118886, 618442)  
4  Cyamemazine may increase the photosensitizing ...   (118886, 21877)  


## 📂 Load RxNorm Mappings

In [3]:
with open('../data/rxnorm_lookups.pkl', 'rb') as f:
    lookups = pickle.load(f)

name_to_rxcui = lookups['name_to_rxcui']
rxcui_to_names = lookups['rxcui_to_names']
bn_to_in_map = lookups['bn_to_in_map']
ingredient_name_to_rxcui = lookups['ingredient_name_to_rxcui']

print(f"✅ Loaded RxNorm mappings:")
print(f"   Name→RxCUI: {len(name_to_rxcui):,}")
print(f"   RxCUI→Names: {len(rxcui_to_names):,}")
print(f"   Brand→Ingredient: {len(bn_to_in_map):,}")
print(f"   Ingredient cache: {len(ingredient_name_to_rxcui):,}")

✅ Loaded RxNorm mappings:
   Name→RxCUI: 157,972
   RxCUI→Names: 82,134
   Brand→Ingredient: 77,518
   Ingredient cache: 6,409


## 🔧 Drug Normalization Function

In [4]:
def normalize_drug(drug_name):
    """Normalize drug name to ingredient RxCUI using ingestion logic"""
    rxcui, dtype = normalize_to_ingredient_rxcui(
        drug_name,
        name_to_rxcui,
        bn_to_in_map,
        ingredient_name_to_rxcui
    )
    return rxcui


print("🔍 Normalization tests:")
test_drugs = ['Tylenol', 'Acetaminophen', 'Advil', 'Ibuprofen', 'Warfarin']
for drug in test_drugs:
    rxcui = normalize_drug(drug)
    print(f"   {drug:15} → RxCUI {rxcui}")

🔍 Normalization tests:
   Tylenol         → RxCUI 161
   Acetaminophen   → RxCUI 161
   Advil           → RxCUI 5640
   Ibuprofen       → RxCUI 5640
   Warfarin        → RxCUI 11289


## 🧠 Build or Load FAISS index (AWS Bedrock Titan V2)

In [5]:
# Initialize AWS Bedrock client
os.environ['AWS_SHARED_CREDENTIALS_FILE'] = "C:\\Users\\USER\\AI-ML-PATHWAYS\\MLOps\\aws-sdk\\.aws-ng\\credentials"
os.environ['AWS_CONFIG_FILE'] = "C:\\Users\\USER\\AI-ML-PATHWAYS\\MLOps\\aws-sdk\\.aws-ng\\config"

session_ng = boto3.Session()
bedrock_client = session_ng.client("bedrock-runtime", region_name="us-east-1")

print("✅ AWS Bedrock client initialized")

# Initialize GPT-5 for reasoning (still using OpenAI for LLM)
load_dotenv("../.env")
gpt_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Paths for AWS Bedrock version
index_path = Path("../data/aws_faiss_index.bin")
embeddings_path = Path("../data/aws_interaction_embeddings.npy")


def get_bedrock_embeddings(texts, dimensions=512):
    """
    Generate embeddings using Amazon Titan Text Embeddings V2
    Optimized for medical RAG with 512 dimensions
    """
    embeddings_list = []

    if not isinstance(texts, list):
        texts = [texts]

    for idx, text in enumerate(texts, 1):
        payload = {
            "inputText": text,
            "dimensions": dimensions,
            "normalize": True  # Optimized for cosine similarity
        }

        print(f"\rEmbedding {idx}/{len(texts)}...", end="", flush=True)

        response = bedrock_client.invoke_model(
            modelId="amazon.titan-embed-text-v2:0",
            body=json.dumps(payload)
        )
        body = json.loads(response["body"].read())
        embeddings_list.append(body["embedding"])

        time.sleep(0.05)  # Rate limiting

    print()
    return np.array(embeddings_list, dtype="float32")


if index_path.exists() and embeddings_path.exists():
    print("Loading existing FAISS index (AWS Bedrock Titan V2)...")
    index = faiss.read_index(str(index_path))
    embeddings = np.load(str(embeddings_path))
    print(f"Loaded index with {index.ntotal:,} vectors ({embeddings.shape[1]} dims)")

else:
    print("Building FAISS index with AWS Bedrock Titan V2...")
    print("   Resume-safe: automatically recovers from interruptions")

    texts = [
        f"{row['Drug 1']} and {row['Drug 2']} interaction: {row['Interaction Description']}"
        for _, row in kb_df.iterrows()
    ]
    print(f"   Generating embeddings for {len(texts):,} interactions...")

    embeddings_list = []
    batch_size = 50
    start_time = time.time()

    # Resume-safe setup
    partial_path = embeddings_path.with_suffix('.partial.npy')
    completed = 0
    if partial_path.exists():
        existing = np.load(partial_path)
        completed = existing.shape[0]
        embeddings_list.append(existing)
        print(f"Resuming from batch {completed // batch_size:,} ({completed:,} embeddings saved)")

    # Generate embeddings with periodic checkpoints
    for i in range(completed, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_embeddings = get_bedrock_embeddings(batch, dimensions=512)
        embeddings_list.append(batch_embeddings)

        # Save checkpoint every 5 batches
        if ((i // batch_size) + 1) % 5 == 0:
            temp = np.vstack(embeddings_list)
            np.save(partial_path, temp)
            print(f"Checkpoint saved: {temp.shape[0]:,} embeddings")

        if (i + batch_size) % 500 == 0:
            elapsed = time.time() - start_time
            rate = (i + batch_size) / elapsed
            remaining = (len(texts) - i - batch_size) / rate / 60
            print(f"      Progress: {i + batch_size:,}/{len(texts):,} ({remaining:.1f} min remaining)")

    embeddings = np.vstack(embeddings_list)
    print(f"Generated {embeddings.shape[0]:,} embeddings ({embeddings.shape[1]} dimensions)")

    # Build FAISS index (cosine similarity via inner product)
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    # Save artifacts
    index_path.parent.mkdir(exist_ok=True)
    faiss.write_index(index, str(index_path))
    np.save(str(embeddings_path), embeddings)

    # Clean up partial file
    if partial_path.exists():
        os.remove(partial_path)

    total_min = (time.time() - start_time) / 60
    print(f"Saved FAISS index ({index.ntotal:,} vectors) in {total_min:.1f} min")

✅ AWS Bedrock client initialized
Loading existing FAISS index (AWS Bedrock Titan V2)...
Loaded index with 170,782 vectors (512 dims)


## 🎯 Tier 1: Direct KB Lookup

In [6]:
def direct_lookup_dynamic(drug_input):
    """Direct KB lookup via RxCUI pair matching"""
    queries = [drug_input] if isinstance(drug_input, str) else drug_input
    all_results = []

    for query_text in queries:
        clean_query = query_text.lower().replace("interaction", "").strip()

        if " and " in clean_query:
            parts = clean_query.split(" and ")
            drug1 = parts[0].strip()
            drug2 = parts[1].strip().split()[0]
        else:
            print(f"   ⚠️ Invalid format: {query_text}")
            all_results.append({"query": query_text, "hits": [], "tier": 1})
            continue

        rxcui1 = normalize_drug(drug1)
        rxcui2 = normalize_drug(drug2)

        if not rxcui1 or not rxcui2:
            print(f"   ⚠️ Could not normalize: {drug1} or {drug2}")
            all_results.append({"query": query_text, "hits": [], "tier": 1})
            continue

        pair_key = tuple(sorted([rxcui1, rxcui2]))
        match = kb_df[kb_df['pair_key'] == pair_key]

        hits = []
        if not match.empty:
            hits.append({
                'drug1': match['Drug 1'].values[0],
                'drug2': match['Drug 2'].values[0],
                'evidence': match['Interaction Description'].values[0],
                'retrieval_score': 1.0
            })
            print(f"Direct match: {drug1} + {drug2}")
        else:
            print(f"No direct match: {drug1} + {drug2}")

        all_results.append({"query": query_text, "hits": hits, "tier": 1})

    return all_results

## 🔎 Tier 2: FAISS Semantic Search

In [7]:
def semantic_search_dynamic(query_input, k=5):
    """Semantic search using AWS Bedrock embeddings"""
    CONFIDENCE_THRESHOLD = 0.6
    FILTER_SIZE_LIMIT = 1000

    queries = [query_input] if isinstance(query_input, str) else query_input
    all_results = []

    for query_text in queries:
        # Generate query embedding using Bedrock
        query_embedding = get_bedrock_embeddings([query_text], dimensions=512)
        query_embedding = query_embedding / (np.linalg.norm(query_embedding, axis=1, keepdims=True) + 1e-8)

        # Lexical pre-filtering
        mask = None
        if " and " in query_text.lower():
            parts = query_text.split(" and ")
            drug1 = parts[0].strip()
            drug2 = parts[1].split()[0].strip()
            mask = (
                    kb_df['Drug 1'].str.contains(drug1, case=False, na=False) |
                    kb_df['Drug 2'].str.contains(drug1, case=False, na=False) |
                    kb_df['Drug 1'].str.contains(drug2, case=False, na=False) |
                    kb_df['Drug 2'].str.contains(drug2, case=False, na=False)
            )

        if mask is not None and mask.sum() > 0 and mask.sum() <= FILTER_SIZE_LIMIT:
            filtered_df = kb_df[mask]
            subset_embeddings = embeddings[filtered_df.index.values]
            index_subset = faiss.IndexFlatIP(subset_embeddings.shape[1])
            index_subset.add(subset_embeddings)
        else:
            filtered_df = kb_df
            index_subset = index

        distances, indices = index_subset.search(query_embedding, k)
        similarities = distances[0]

        hits = []
        for idx, score in zip(indices[0], similarities):
            if score >= CONFIDENCE_THRESHOLD:
                row = filtered_df.iloc[idx]
                hits.append({
                    "drug1": row['Drug 1'],
                    "drug2": row['Drug 2'],
                    "evidence": row['Interaction Description'],
                    "retrieval_score": float(score)
                })

        all_results.append({"query": query_text, "hits": hits, "tier": 2})

    return all_results

## 🔄 Unified Hybrid Retrieval (Tier 1 → Tier 2 → Tier 3)

In [8]:
def check_drug_interaction(drug_input, k=5):
    """Tiered retrieval: direct lookup → semantic search → no evidence"""
    CONFIDENCE_THRESHOLD = 0.6
    FILTER_SIZE_LIMIT = 1000

    queries = [drug_input] if isinstance(drug_input, str) else drug_input
    all_results = []

    for query_text in queries:
        clean_query = query_text.lower().replace("interaction", "").strip()

        if " and " not in clean_query:
            print(f"   ⚠️ Invalid format: {query_text}")
            all_results.append({"query": query_text, "hits": [], "tier": 3})
            continue

        parts = clean_query.split(" and ")
        drug1 = parts[0].strip()
        drug2 = parts[1].strip().split()[0]

        rxcui1 = normalize_drug(drug1)
        rxcui2 = normalize_drug(drug2)

        # Tier 1: Direct lookup
        if rxcui1 and rxcui2:
            pair_key = tuple(sorted([rxcui1, rxcui2]))
            match = kb_df[kb_df['pair_key'] == pair_key]

            if not match.empty:
                hits = [{
                    'drug1': match['Drug 1'].values[0],
                    'drug2': match['Drug 2'].values[0],
                    'evidence': match['Interaction Description'].values[0],
                    'retrieval_score': 1.0
                }]
                print(f"Tier 1: Direct match for {drug1} + {drug2}")
                all_results.append({"query": query_text, "hits": hits, "tier": 1})
                continue
            else:
                print(f"Tier 1: No direct match, trying semantic search...")
        else:
            print(f"   ⚠️ Could not normalize via RxCUI, trying semantic search...")

        # Tier 2: Semantic search with AWS Bedrock
        print(f"Tier 2: Semantic search for {drug1} + {drug2} (AWS Bedrock)...")

        query_embedding = get_bedrock_embeddings([f"{drug1} and {drug2} interaction"], dimensions=512)
        query_embedding = query_embedding / (np.linalg.norm(query_embedding, axis=1, keepdims=True) + 1e-8)

        mask = (
                kb_df['Drug 1'].str.contains(drug1, case=False, na=False) |
                kb_df['Drug 2'].str.contains(drug1, case=False, na=False) |
                kb_df['Drug 1'].str.contains(drug2, case=False, na=False) |
                kb_df['Drug 2'].str.contains(drug2, case=False, na=False)
        )

        if mask.sum() > 0 and mask.sum() <= FILTER_SIZE_LIMIT:
            filtered_df = kb_df[mask]
            subset_embeddings = embeddings[filtered_df.index.values]
            index_subset = faiss.IndexFlatIP(subset_embeddings.shape[1])
            index_subset.add(subset_embeddings)
            print(f"      Using filtered index: {len(filtered_df)} interactions")
        else:
            filtered_df = kb_df
            index_subset = index
            print(f"      Using global index")

        distances, indices = index_subset.search(query_embedding, k)
        similarities = distances[0]

        hits = []
        for idx, score in zip(indices[0], similarities):
            if score >= CONFIDENCE_THRESHOLD:
                row = filtered_df.iloc[idx]
                hits.append({
                    "drug1": row['Drug 1'],
                    "drug2": row['Drug 2'],
                    "evidence": row['Interaction Description'],
                    "retrieval_score": float(score)
                })

        if hits:
            print(f"Tier 2: Found {len(hits)} semantic matches")
            all_results.append({"query": query_text, "hits": hits, "tier": 2})
        else:
            print(f"Tier 3: No evidence found")
            all_results.append({"query": query_text, "hits": [], "tier": 3})

    return all_results

## 💊 Polypharmacy Analyzer (N-Drug Pairwise Expansion)

In [9]:
def check_polypharmacy(drug_input):
    """Handle multi-drug queries with pairwise expansion"""
    if isinstance(drug_input, str):
        text = drug_input.replace("interaction", "").strip()
        if " and " in text:
            parts = text.split(" and ")
            first_part = parts[0]
            last_drug = parts[1].strip()
            drugs = [d.strip() for d in first_part.split(",") if d.strip()] + [last_drug]
        else:
            drugs = [d.strip() for d in text.split(",") if d.strip()]
    else:
        drugs = drug_input

    if len(drugs) < 2:
        return {'error': 'Need at least 2 drugs'}

    pairs = list(combinations(drugs, 2))
    queries = [f"{d1} and {d2} interaction" for d1, d2 in pairs]

    print(f"\n🔍 Checking {len(pairs)} drug pairs...")
    results = check_drug_interaction(queries)

    return {'drugs': drugs, 'num_pairs': len(pairs), 'results': results}

In [10]:
check_polypharmacy("Ibuprofen and Warfarin interaction")


🔍 Checking 1 drug pairs...
Tier 1: Direct match for ibuprofen + warfarin


{'drugs': ['Ibuprofen', 'Warfarin'],
 'num_pairs': 1,
 'results': [{'query': 'Ibuprofen and Warfarin interaction',
   'hits': [{'drug1': 'Warfarin',
     'drug2': 'Ibuprofen',
     'evidence': 'Warfarin may increase the anticoagulant activities of Ibuprofen.',
     'retrieval_score': 1.0}],
   'tier': 1}]}

In [11]:
check_polypharmacy("Warfarin, Aspirin and Acetaminophen interaction")


🔍 Checking 3 drug pairs...
Tier 1: Direct match for warfarin + aspirin
Tier 1: Direct match for warfarin + acetaminophen
Tier 1: No direct match, trying semantic search...
Tier 2: Semantic search for aspirin + acetaminophen (AWS Bedrock)...
Embedding 1/1...
      Using filtered index: 217 interactions
   ✅ Tier 2: Found 2 semantic matches


{'drugs': ['Warfarin', 'Aspirin', 'Acetaminophen'],
 'num_pairs': 3,
 'results': [{'query': 'Warfarin and Aspirin interaction',
   'hits': [{'drug1': 'Warfarin',
     'drug2': 'Acetylsalicylic acid',
     'evidence': 'Warfarin may increase the anticoagulant activities of Acetylsalicylic acid.',
     'retrieval_score': 1.0}],
   'tier': 1},
  {'query': 'Warfarin and Acetaminophen interaction',
   'hits': [{'drug1': 'Warfarin',
     'drug2': 'Acetaminophen',
     'evidence': 'Warfarin may increase the anticoagulant activities of Acetaminophen.',
     'retrieval_score': 1.0}],
   'tier': 1},
  {'query': 'Aspirin and Acetaminophen interaction',
   'hits': [{'drug1': 'Acetaminophen',
     'drug2': 'Isradipine',
     'evidence': 'The metabolism of Isradipine can be decreased when combined with Acetaminophen.',
     'retrieval_score': 0.6239258646965027},
    {'drug1': 'Acetaminophen',
     'drug2': 'Desipramine',
     'evidence': 'The metabolism of Desipramine can be decreased when combined 

In [12]:
check_polypharmacy("blood thinner and pain medication interaction")


🔍 Checking 1 drug pairs...
   ⚠️ Could not normalize via RxCUI, trying semantic search...
Tier 2: Semantic search for blood thinner + pain (AWS Bedrock)...
Embedding 1/1...
      Using global index
   ℹ️ Tier 3: No evidence found


{'drugs': ['blood thinner', 'pain medication'],
 'num_pairs': 1,
 'results': [{'query': 'blood thinner and pain medication interaction',
   'hits': [],
   'tier': 3}]}

## 🧪 Pipeline Tests

In [13]:
print("\n" + "=" * 70)
print("PIPELINE VALIDATION (AWS BEDROCK TITAN V2)")
print("=" * 70)

print("\n1️⃣ Single pair (direct match):")
result = check_drug_interaction("Warfarin and Aspirin interaction")
print(f"   Tier: {result[0]['tier']}, Hits: {len(result[0]['hits'])}")

print("\n2️⃣ Polypharmacy (3 drugs = 3 pairs):")
result = check_polypharmacy("Warfarin, Aspirin and Ibuprofen")
print(f"   Pairs: {result['num_pairs']}, Results: {len(result['results'])}")

print("\n3️⃣ Semantic search fallback (AWS Bedrock):")
result = check_drug_interaction("blood thinner and pain medication interaction")
print(f"   Tier: {result[0]['tier']}, Hits: {len(result[0]['hits'])}")

print("\n✅ AWS Bedrock pipeline ready for safety.py integration!")


PIPELINE VALIDATION (AWS BEDROCK TITAN V2)

1️⃣ Single pair (direct match):
Tier 1: Direct match for warfarin + aspirin
   Tier: 1, Hits: 1

2️⃣ Polypharmacy (3 drugs = 3 pairs):

🔍 Checking 3 drug pairs...
Tier 1: Direct match for warfarin + aspirin
Tier 1: Direct match for warfarin + ibuprofen
Tier 1: Direct match for aspirin + ibuprofen
   Pairs: 3, Results: 3

3️⃣ Semantic search fallback (AWS Bedrock):
   ⚠️ Could not normalize via RxCUI, trying semantic search...
Tier 2: Semantic search for blood thinner + pain (AWS Bedrock)...
Embedding 1/1...
      Using global index
   ℹ️ Tier 3: No evidence found
   Tier: 3, Hits: 0

✅ AWS Bedrock pipeline ready for safety.py integration!
